# Broad synthetic validation

This small example compares the published-style EACF, AsteroScale-restricted EACF, and joint Urdr detector using paired but independently evaluated simulations. Scientific validation should use a much larger pre-registered matrix and more realisations.

In [ ]:
import numpy as np

from urdr import (
    CoherentSignalConfig,
    SegmentSystematicConfig,
    SimulationConfig,
    ValidationCase,
    benchmark_synthetic_validation,
    make_observing_window,
)

In [ ]:
window = make_observing_window(
    duration_days=0.8,
    cadence_seconds=120.0,
    gaps_days=((0.39, 0.41),),
)
simulation = SimulationConfig(
    white_noise_sigma=0.2,
    granulation_amplitude=0.1,
    numax_uhz=1000.0,
    delta_nu_uhz=100.0,
    envelope_width_uhz=400.0,
    oscillation_amplitude=1.8,
)

In [ ]:
case = ValidationCase(
    name="held_out_gap",
    split="validation",
    window=window,
    simulation=simulation,
    published_centres_uhz=np.linspace(500.0, 1500.0, 11),
    restricted_centres_uhz=np.linspace(700.0, 1300.0, 7),
    filter_width_uhz=500.0,
    delta_nu_grid_uhz=np.array([90.0, 100.0, 110.0]),
    segments_days=((0.0, 0.4), (0.4, 0.8)),
    coherent_contaminants={
        "single_line": CoherentSignalConfig(1000.0, 0.8),
    },
    segment_systematics={
        "variance_jump": [
            SegmentSystematicConfig(0.4, 0.8, amplitude_scale=4.0)
        ],
    },
    max_lag_seconds=25_000.0,
)

In [ ]:
result = benchmark_synthetic_validation(
    [case],
    calibration_realizations=16,
    evaluation_realizations=4,
    validation_fraction=0.25,
    target_false_positive_rate=0.25,
    reliability_bins=4,
    seed=42,
)
result.to_records()

In [ ]:
[vars(row) for row in result.reliability]

The tutorial uses a deliberately coarse 25 per cent false-positive target because only four held-out joint-calibration realisations are available. A 1 per cent scientific target requires at least 100 held-out realisations, and preferably substantially more. Reliability and detection performance should be inspected separately.